# University Knowledge Assistant — RAG Pipeline Walkthrough

A context-aware university knowledge assistant built using **Retrieval-Augmented
Generation (RAG)**. This notebook walks through the full pipeline step by step:

1. **Ingestion** — load university documents, extract text, clean it, split into chunks
2. **Embedding & Storage** — convert chunks into vector embeddings, store in ChromaDB
3. **Retrieval** — given a question, find the most relevant chunks
4. **Generation** — pass retrieved chunks to an LLM (Claude) with a grounded system
   prompt, so it answers *only* from your documents instead of hallucinating

> This notebook mirrors the same logic as the standalone `app.py` / `ingest.py` /
> `rag_pipeline.py` project — use this notebook to explore, test, and evaluate the
> pipeline interactively; use the `.py` files + `streamlit run app.py` for the actual
> deployed chat app.


## 0. Setup

Install dependencies and set your Anthropic API key. Get a key from
https://console.anthropic.com/


In [ ]:
%pip install -q chromadb sentence-transformers pypdf anthropic python-dotenv

In [ ]:
import os
import re
import glob
import uuid

import chromadb
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from anthropic import Anthropic

# Set your API key here, or via environment variable / .env file
os.environ.setdefault("ANTHROPIC_API_KEY", "")  # <-- paste your key here if not using env vars

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
print("API key configured:", bool(ANTHROPIC_API_KEY))


## 1. Configuration

All the tunable settings for the pipeline in one place.


In [ ]:
DOCS_DIR = "sample_docs"          # folder containing your PDF/TXT documents
CHROMA_DIR = "chroma_db"           # where the vector database is persisted
COLLECTION_NAME = "university_knowledge_base"

CHUNK_SIZE = 800                   # characters per chunk
CHUNK_OVERLAP = 150                # overlap between consecutive chunks

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"   # free, local, no API key needed

TOP_K = 4                          # number of chunks retrieved per question

LLM_MODEL = "claude-sonnet-5"
LLM_MAX_TOKENS = 1024
LLM_TEMPERATURE = 0.2              # low temperature: stay grounded, not creative

os.makedirs(DOCS_DIR, exist_ok=True)
os.makedirs(CHROMA_DIR, exist_ok=True)


## 2. Get a sample document

If you already have university PDFs, drop them into the `sample_docs/` folder and
skip this cell. Otherwise, this cell writes a sample student handbook so you can run
the whole pipeline immediately.


In [ ]:
sample_text = """GREENFIELD UNIVERSITY - STUDENT HANDBOOK 2026

SECTION 1: ATTENDANCE POLICY
Students must maintain a minimum of 75% attendance in each registered course to be
eligible to sit for the end-semester examination. Students falling between 65% and 75%
attendance may be granted condonation by the Head of Department on submission of a
valid medical certificate or supporting documentation. Students below 65% attendance
will be debarred from the examination and must repeat the course in a subsequent
semester.

SECTION 2: EXAMINATION GUIDELINES
End-semester examinations are conducted over a 15-day window at the end of each
semester. Students must carry their university ID card to every examination. Electronic
devices, including smartwatches and mobile phones, are strictly prohibited in the
examination hall. Any student found using unfair means will be subject to disciplinary
action ranging from cancellation of that paper to debarment for one full academic year.
Re-evaluation requests must be submitted within 7 days of result declaration.

SECTION 3: GRADING SYSTEM
The university follows a 10-point Cumulative Grade Point Average (CGPA) system.
Grades are awarded as follows: O (90-100, 10 points), A+ (80-89, 9 points), A (70-79, 8
points), B+ (60-69, 7 points), B (50-59, 6 points), C (40-49, 5 points), and F (below 40,
0 points, indicating failure). A minimum CGPA of 5.0 is required to graduate.

SECTION 4: PLACEMENT AND INTERNSHIP POLICY
All final-year students are required to complete a minimum 8-week internship as part of
their degree requirements. The Placement Cell coordinates with partner companies to
offer internship opportunities starting in the sixth semester. A minimum CGPA of 6.0 is
required to participate in on-campus placement drives.

SECTION 5: LIBRARY RULES
The central library is open from 8:00 AM to 10:00 PM on all working days and 9:00 AM
to 5:00 PM on weekends. Undergraduate students may borrow up to 4 books at a time
for a period of 14 days. A fine of Rs. 5 per day is charged for overdue books.

SECTION 6: SCHOLARSHIPS AND FINANCIAL AID
Merit scholarships covering up to 50% of tuition fees are awarded to students in the top
5% of their cohort each semester, based on CGPA. Students availing any scholarship
must maintain a minimum 7.0 CGPA to continue receiving it.

SECTION 7: CODE OF CONDUCT
Plagiarism in assignments, projects, or reports will result in a zero grade for that
submission on the first offense. Use of generative AI tools for assignments must be
disclosed where required by the course instructor; undisclosed use may be treated as
academic dishonesty.
"""

with open(os.path.join(DOCS_DIR, "student_handbook.txt"), "w") as f:
    f.write(sample_text)

print("Sample document written to sample_docs/student_handbook.txt")


## 3. Load documents

Collect every PDF/TXT file in `sample_docs/` and extract raw text from each.


In [ ]:
def extract_text_from_pdf(path: str) -> str:
    reader = PdfReader(path)
    pages_text = [page.extract_text() or "" for page in reader.pages]
    return "\n".join(pages_text)


def load_documents(docs_dir: str) -> list[dict]:
    documents = []
    file_paths = glob.glob(os.path.join(docs_dir, "*.pdf")) + \
                 glob.glob(os.path.join(docs_dir, "*.txt"))

    for path in file_paths:
        filename = os.path.basename(path)
        if path.lower().endswith(".pdf"):
            text = extract_text_from_pdf(path)
        else:
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                text = f.read()
        if text.strip():
            documents.append({"source": filename, "text": text})
            print(f"Loaded: {filename} ({len(text)} chars)")
    return documents


documents = load_documents(DOCS_DIR)
print(f"\nTotal documents loaded: {len(documents)}")


## 4. Clean text

Strip stray characters, collapse repeated whitespace and blank lines.


In [ ]:
def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


for doc in documents:
    doc["text"] = clean_text(doc["text"])

print("Cleaned", len(documents), "document(s)")


## 5. Chunk documents

Split each document into overlapping chunks. Overlap preserves context across chunk
boundaries; breaking on sentence boundaries (where possible) avoids cutting sentences
in half.


In [ ]:
def chunk_text(text: str, chunk_size: int, overlap: int) -> list[str]:
    if len(text) <= chunk_size:
        return [text]

    chunks = []
    start = 0
    text_len = len(text)

    while start < text_len:
        end = min(start + chunk_size, text_len)
        if end < text_len:
            boundary = text.rfind(". ", start, end)
            if boundary != -1 and boundary > start + chunk_size * 0.5:
                end = boundary + 1
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= text_len:
            break
        start = end - overlap

    return chunks


all_chunks, all_metadatas, all_ids = [], [], []
for doc in documents:
    chunks = chunk_text(doc["text"], CHUNK_SIZE, CHUNK_OVERLAP)
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        all_metadatas.append({"source": doc["source"], "chunk_index": i})
        all_ids.append(str(uuid.uuid4()))
    print(f"{doc['source']}: {len(chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")
print("\nExample chunk:\n---")
print(all_chunks[0])


## 6. Generate embeddings

Use a free, local sentence-transformer model to convert each chunk into a vector.
No API key or cost required for this step — it runs on CPU.


In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
embeddings = embedding_model.encode(all_chunks, show_progress_bar=True).tolist()

print(f"Generated {len(embeddings)} embeddings, each of dimension {len(embeddings[0])}")


## 7. Store in ChromaDB

Persist the chunks, their embeddings, and metadata (source file + chunk index) in a
local vector database so they can be searched later.


In [ ]:
client = chromadb.PersistentClient(path=CHROMA_DIR)

# Start fresh each run so re-ingesting doesn't duplicate chunks
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = client.create_collection(COLLECTION_NAME)
collection.add(
    ids=all_ids,
    embeddings=embeddings,
    documents=all_chunks,
    metadatas=all_metadatas,
)

print(f"Stored {collection.count()} chunks in ChromaDB at '{CHROMA_DIR}'")


## 8. Retrieval

Given a question, embed it with the same model and find the most similar chunks in
the vector database.


In [ ]:
def retrieve(question: str, top_k: int = TOP_K) -> list[dict]:
    query_embedding = embedding_model.encode([question]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=top_k)

    retrieved = []
    docs = results.get("documents", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    distances = results.get("distances", [[]])[0]

    for doc, meta, dist in zip(docs, metas, distances):
        retrieved.append({
            "text": doc,
            "source": meta.get("source", "unknown"),
            "chunk_index": meta.get("chunk_index", -1),
            "relevance": round(1 - dist, 3),
        })
    return retrieved


# Quick test
test_results = retrieve("What is the attendance policy?")
for r in test_results:
    print(f"[{r['source']} · chunk {r['chunk_index']} · relevance {r['relevance']}]")
    print(r["text"][:150], "...\n")


## 9. Grounded generation

Pass the retrieved chunks to Claude along with a system prompt that **forces it to
answer only from the provided context** — this is what prevents hallucination. If the
answer isn't in the retrieved chunks, the model is instructed to say so rather than
guess.


In [ ]:
SYSTEM_PROMPT = """You are the University Knowledge Assistant. You answer student
questions using ONLY the context passages provided below, which come from official
university documents (handbooks, regulations, exam guidelines, course material, etc.).

Rules:
- Base your answer strictly on the provided context. Do not use outside knowledge.
- If the context does not contain the answer, say clearly: "I couldn\'t find this in
  the available university documents." Do not guess or fabricate policy details.
- Be concise and direct. Use bullet points for lists (e.g. steps, requirements).
- When helpful, mention which document the information came from.
"""

llm_client = Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None


def generate_answer(question: str, retrieved_chunks: list[dict]) -> str:
    if not retrieved_chunks:
        return "I couldn\'t find this in the available university documents."

    context_block = "\n\n".join(
        f"[Source: {c['source']}, chunk {c['chunk_index']}]\n{c['text']}"
        for c in retrieved_chunks
    )

    if llm_client is None:
        return ("(LLM not configured — set ANTHROPIC_API_KEY to enable grounded "
                "answers. Showing the most relevant retrieved passage instead.)\n\n"
                + retrieved_chunks[0]["text"])

    response = llm_client.messages.create(
        model=LLM_MODEL,
        max_tokens=LLM_MAX_TOKENS,
        temperature=LLM_TEMPERATURE,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": f"Context:\n{context_block}\n\nQuestion: {question}"}],
    )
    return response.content[0].text


## 10. Full pipeline: ask a question

Combine retrieval + generation into a single function, and try it out.


In [ ]:
def answer(question: str, top_k: int = TOP_K) -> dict:
    chunks = retrieve(question, top_k)
    answer_text = generate_answer(question, chunks)
    return {"question": question, "answer": answer_text, "sources": chunks}


result = answer("What is the minimum attendance required to sit for exams?")
print("Q:", result["question"])
print("\nA:", result["answer"])
print("\nSources used:")
for s in result["sources"]:
    print(f"  - {s['source']} (chunk {s['chunk_index']}, relevance {s['relevance']})")


## 11. Test with more questions

Try a range of in-scope and out-of-scope questions to evaluate the pipeline.
An out-of-scope question should trigger the "I couldn't find this" fallback rather
than a hallucinated answer — that's the key thing to demonstrate for your project
evaluation.


In [ ]:
test_questions = [
    "What happens if I'm caught using unfair means in an exam?",
    "How many books can I borrow from the library?",
    "What CGPA do I need to keep my scholarship?",
    "Is generative AI allowed for assignments?",
    "What is the campus wifi password?",  # out-of-scope: should say "couldn't find"
]

for q in test_questions:
    result = answer(q)
    print("Q:", q)
    print("A:", result["answer"])
    print("-" * 80)


## 12. Evaluation notes

For your project report, worth documenting:

- **Retrieval quality**: check the `relevance` score (1 − cosine distance) for each
  test question — are the top chunks actually on-topic?
- **Grounding behaviour**: the out-of-scope question above should trigger the
  "I couldn't find this in the available university documents" response rather than
  a guessed answer.
- **Chunking trade-off**: try changing `CHUNK_SIZE` / `CHUNK_OVERLAP` in Section 1 and
  re-running Sections 5–7 to see how it affects retrieval relevance.

## Next steps

- Add more documents to `sample_docs/` (your actual university's PDFs) and re-run
  Sections 3–7 to rebuild the knowledge base with real content.
- For an interactive chat interface instead of this notebook, use the companion
  `app.py` (Streamlit) from the full project package.
